### P-dispersion

The p-dispersion problem consists of locating $p$ facilities in such a way that the minimum distance between any pair of selected facilities is as large as possible. Let $d_{ij}$ denote the distance between node $i$ and node $j$.

Finally, we define the following decision variable:

$$
y_i =
\begin{cases}
1 & \text{if a facility is located at candidate site } i, \\
0 & \text{otherwise.}
\end{cases}
$$

By introducing an auxiliary variable $r$, the problem can be linearized as follows:

$$
\begin{aligned}
\text{max} \quad & r \\
\text{subject to} \quad
& r \leq d_{ij} + M(2 - y_i - y_j) \quad \forall i \in I,\; j \in J: i < j \\
& \sum_{i \in I} y_i = p \\
& y_i \in \{0,1\} \quad \forall i \in I
\end{aligned}
$$

In [ ]:
%pip install -q amplpy numpy matplotlib pandas networkx folium
from amplpy import AMPL, ampl_notebook
import numpy as np

# HiGHS is the default. Gurobi requires an AMPL-compatible license.
SOLVER = "highs"  # or "gurobi"
LICENSE_UUID = "default"  # Colab Community Edition; use your UUID locally
runtime = ampl_notebook(modules=[SOLVER], license_uuid=LICENSE_UUID)

def new_ampl():
    return AMPL()

def solve_checked(model):
    model.solve(solver=SOLVER)
    if model.solve_result != "solved":
        raise RuntimeError(f"No proven optimal solution: {model.solve_result}. "
                           "Inspect the solver log before extracting values.")

def values(model, name):
    # Numeric dictionaries keep plotting independent of the solver API.
    return model.var[name].get_values().to_dict()




In [ ]:
# Number of candidate facility sites
num_facilities = 5

# Number of customers
num_customers = 5

# Sets
I = range(num_facilities)
J = range(num_customers)

# Symmetric service cost matrix
c = np.array([[0, 3, 3, 6, 3],
              [3, 0, 4, 5, 5],
              [3, 4, 0, 2, 4],
              [6, 5, 2, 0, 5],
              [3, 5, 4, 5, 0]])

# Customer demand
d = np.array([10, 8, 5, 8, 12])

# Number of facilities to locate
p = 2

# A big number
M = float(np.max(c)) + 1

In [ ]:
m = new_ampl()
m.eval(r"""
set I ordered;
param distance {I,I};
param p integer >= 2;
param M >= 0;
var y {I} binary;
var r >= 0;
maximize Total_Cost: r;
subject to Separation {i in I,j in I: ord(i)<ord(j)}:
    r <= distance[i,j] + M*(2-y[i]-y[j]);
subject to Cardinality: sum {i in I} y[i] = p;
""")
m.set["I"] = list(I)
m.param["distance"] = {(i,j): float(c[i,j]) for i in I for j in I}
m.param["p"] = p
m.param["M"] = M

solve_checked(m)
y = values(m, "y")
r = m.var["r"].value()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def solve_and_plot_p_dispersion(n=5, p=3, seed=42, coord_range=(0, 100), metric="euclidean"):

    rng = np.random.default_rng(seed)
    I = range(n)

    points = rng.uniform(coord_range[0], coord_range[1], size=(n, 2))

    d = np.zeros((n, n), dtype=float)
    for i in I:
        for j in I:
            if i == j:
                d[i, j] = 0.0
            elif i < j:
                dx = points[i, 0] - points[j, 0]
                dy = points[i, 1] - points[j, 1]
                if metric.lower() == "euclidean":
                    dist = float(np.sqrt(dx * dx + dy * dy))
                elif metric.lower() == "manhattan":
                    dist = float(abs(dx) + abs(dy))
                else:
                    raise ValueError("metric must be 'euclidean' or 'manhattan'")
                d[i, j] = d[j, i] = dist

    # Big-M
    M = float(np.max(d))

    m = new_ampl()
    m.eval(r"""
    set I ordered;
    param distance {I,I};
    param p integer >= 2;
    param M >= 0;
    var y {I} binary;
    var r >= 0;
    maximize Total_Cost: r;
    subject to Separation {i in I,j in I: ord(i)<ord(j)}:
        r <= distance[i,j] + M*(2-y[i]-y[j]);
    subject to Cardinality: sum {i in I} y[i] = p;
    """)
    m.set["I"] = list(I)
    m.param["distance"] = {(i,j): float(d[i,j]) for i in I for j in I}
    m.param["p"] = p
    m.param["M"] = M
    
    solve_checked(m)
    y = values(m, "y")
    r = m.var["r"].value()

    selected_points = [i for i in I if y[i] > 0.5]
    print(f"Selected points: {selected_points}")
    print(f"Minimum distance among selected points (r): {r:.4f}")

    fig, ax = plt.subplots(figsize=(8, 7))

    # All points
    ax.scatter(points[:, 0], points[:, 1], marker="o")
    for i in I:
        ax.annotate(f"P{i}", (points[i, 0], points[i, 1]), xytext=(5, 5), textcoords="offset points")

    # Selected points (highlighted)
    if selected_points:
        ax.scatter(points[selected_points, 0], points[selected_points, 1], marker="s")

        # Lines between selected pairs
        for a_idx in range(len(selected_points)):
            for b_idx in range(a_idx + 1, len(selected_points)):
                i = selected_points[a_idx]
                j = selected_points[b_idx]
                ax.plot([points[i, 0], points[j, 0]], [points[i, 1], points[j, 1]], linewidth=1)

        # Show the pair that defines the minimum distance (if p >= 2)
        if len(selected_points) >= 2:
            best_pair = None
            best_val = float("inf")
            for a_idx in range(len(selected_points)):
                for b_idx in range(a_idx + 1, len(selected_points)):
                    i = selected_points[a_idx]
                    j = selected_points[b_idx]
                    if d[i, j] < best_val:
                        best_val = d[i, j]
                        best_pair = (i, j)
            i, j = best_pair
            midx = (points[i, 0] + points[j, 0]) / 2
            midy = (points[i, 1] + points[j, 1]) / 2
            ax.annotate(f"min={best_val:.2f}", (midx, midy), xytext=(5, 5), textcoords="offset points")

    ax.set_title(f"P-dispersion (p={p}) - r={r:.4f} - metric: {metric}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.axis("equal")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return m, d, selected_points, points



In [ ]:
model, distances, selected, pts = solve_and_plot_p_dispersion(n=60, p=8, seed=10)
